# Use `ibm-watsonx-ai` to deploy R script
This notebook contains steps and code to demonstrate how to deploy a R script with the [ibm-watsonx-ai](https://pypi.python.org/pypi/ibm-watsonx-ai) library available in the PyPI repository. This notebook consists of steps to create a R script, create a deployment, create and run a job. 

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Learning goals
The learning goals of this notebook are:

- Create and save a R script.
- Deploy the script using the client library.
- Create and Run a job which utilises the created deployment.

## Contents
This notebook contains the following parts:

1. [Setup](#section1)
2. [Deployment of R Script](#section2)
3. [Create and Run Job](#section3)
4. [Clean up](#section4)
5. [Summary](#section5)

<a id='section1'></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.3",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.3",
    )

#### Create `APIClient` instance

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click New Deployment Space
- Create an empty space
- Go to space `Settings` tab
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.3/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [5]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in Watson Machine Learning, you need to set **space** which you will be using.

In [6]:
client.set.default_space(space_id)

'SUCCESS'

### Software specification

You can use popular tools, libraries, and frameworks to train and deploy machine learning models and functions.

The following list shows the predefined (base) model types and software specifications.

In [7]:
client.software_specifications.list()

,NAME,ID,TYPE,STATE,REPLACEMENT
0,autoai-kb_rt25.1-py3.12,046f5252-0d07-5c23-81ef-18a1a31cdce3,base,supported,
1,tensorflow_rt25.1-py3.12-edt,0747503b-cc2e-5249-9019-59df39a112b8,base,supported,
2,watsonx-textgen-fm-1.0,129aec82-7e65-5c78-b812-4c0a74b916f5,base,supported,
3,masking-flows-spark,13666829-5570-53a7-927b-52d42a101d93,base,supported,
4,autoai-ts_rt25.1-py3.12,1c0c9e1d-8523-520f-867e-102173f16f5e,base,supported,
5,runtime-25.1-r4.4,2b2ddb1d-6c0f-5b95-9d04-3056270ea5c6,base,supported,
6,tensorflow_rt24.1-py3.11,2c33167d-b11c-5490-a305-3e5e95db5c4d,base,supported,
7,pytorch-onnx_rt24.1-py3.11,2da185aa-eac3-59a5-bb8e-0e5b60458a15,base,supported,
8,tensorflow_rt24.1-py3.11-edt,2f53b747-c01a-58a6-a20c-8390ecfd8625,base,supported,
9,kernel-spark3.5-py3.11,30ad99e2-7cfb-511a-9a3e-4cc308071fd3,base,supported,


You can select the software specification using the function below.

In [8]:
base_sw_spec_id = client.software_specifications.get_uid_by_name("runtime-25.1-r4.4")
print(base_sw_spec_id)

2b2ddb1d-6c0f-5b95-9d04-3056270ea5c6


<a id='section2'></a>
## 2. R Script Deployment

### Save R Script

This file will be saved locally so you can deploy and run it later.

In [9]:
%%writefile /tmp/CreateSwSpec_script.R

outfname <- file.path(Sys.getenv("BATCH_OUTPUT_DIR"), "swspec.log")

tryCatch({
  writeLines("ok", outfname)
}, error = function(e) {
  writeLines(paste("fail:", e), outfname)
})

Writing /tmp/CreateSwSpec_script.R


The file should be successfully created. 
To check its content, you can use the command below.

In [10]:
%cat /tmp/CreateSwSpec_script.R


outfname <- file.path(Sys.getenv("BATCH_OUTPUT_DIR"), "swspec.log")

tryCatch({
  writeLines("ok", outfname)
}, error = function(e) {
  writeLines(paste("fail:", e), outfname)
})


### Deployment of R Script

You can store and deploy a R script and get its details by running the code in following cells. 

In [11]:
meta_props = {
    client.script.ConfigurationMetaNames.NAME: "RScript",
    client.script.ConfigurationMetaNames.SOFTWARE_SPEC_UID: base_sw_spec_id,
}

In [12]:
script_path = "/tmp/CreateSwSpec_script.R"
script_details = client.script.store(meta_props, file_path=script_path)
script_id = client.script.get_id(script_details)
print("Created script ", script_id)

Creating Script asset...
SUCCESS
Created script  ac758e93-895d-4408-817e-d0302b00de67


In [13]:
import json

print(json.dumps(script_details, indent=2))

{
  "metadata": {
    "name": "RScript",
    "guid": "ac758e93-895d-4408-817e-d0302b00de67",
    "href": "/v2/assets/ac758e93-895d-4408-817e-d0302b00de67?space_id=e645e918-b707-4e6c-9bbe-7304525fb5a0",
    "asset_type": "script",
    "created_at": "2025-11-24T14:16:07Z",
    "last_updated_at": "2025-11-24T14:16:07Z",
    "space_id": "e645e918-b707-4e6c-9bbe-7304525fb5a0",
    "description": ""
  },
  "entity": {
    "script": {
      "language": {
        "name": "R"
      },
      "software_spec": {
        "base_id": "2b2ddb1d-6c0f-5b95-9d04-3056270ea5c6"
      }
    }
  }
}


In [14]:
deployment_meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "rscript_deployment",
    client.deployments.ConfigurationMetaNames.BATCH: {},
    client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {"name": "S"},
}

deployment_details = client.deployments.create(script_id, deployment_meta_props)
deployment_id = client.deployments.get_id(deployment_details)



######################################################################################

Synchronous deployment creation for id: 'ac758e93-895d-4408-817e-d0302b00de67' started

######################################################################################


ready.


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='16ffbd10-3420-41be-98c3-d2ffd975a978'
-----------------------------------------------------------------------------------------------




In [15]:
print(json.dumps(deployment_details, indent=2))

{
  "entity": {
    "asset": {
      "id": "ac758e93-895d-4408-817e-d0302b00de67"
    },
    "batch": {},
    "chat_enabled": false,
    "custom": {},
    "deployed_asset_type": "r_script",
    "hardware_spec": {
      "name": "S",
      "num_nodes": 1
    },
    "name": "rscript_deployment",
    "space_id": "e645e918-b707-4e6c-9bbe-7304525fb5a0",
    "status": {
      "state": "ready"
    }
  },
  "metadata": {
    "created_at": "2025-11-24T14:16:24.964Z",
    "id": "16ffbd10-3420-41be-98c3-d2ffd975a978",
    "modified_at": "2025-11-24T14:16:24.964Z",
    "name": "rscript_deployment",
    "owner": "1000330999",
    "space_id": "e645e918-b707-4e6c-9bbe-7304525fb5a0"
  }
}


<a id='section3'></a>
## 3. Create and Run job

Run the following cells to create and run a job with the deployed script.

In [16]:
def poll_async_job(client: APIClient, job_uid: str) -> dict:
    import time

    while True:
        job_status = client.deployments.get_job_status(job_uid)
        print(job_status)
        state = job_status["state"]
        if state == "completed" or "fail" in state:
            return client.deployments.get_job_details(job_uid)
        time.sleep(5)

In [17]:
job_payload_ref = {
    client.deployments.ScoringMetaNames.OUTPUT_DATA_REFERENCE: {
        "type": "data_asset",
        "location": {"name": "deploy_test_r_script-out"},
    }
}

job = client.deployments.create_job(deployment_id, meta_props=job_payload_ref)
job_id = client.deployments.get_job_uid(job)

In [18]:
job_details = poll_async_job(client, job_id)

{'completed_at': '', 'running_at': '', 'state': 'queued'}
{'completed_at': '', 'running_at': '', 'state': 'queued'}
{'completed_at': '', 'running_at': '', 'state': 'queued'}
{'completed_at': '', 'running_at': '', 'state': 'queued'}
{'completed_at': '', 'running_at': '', 'state': 'queued'}
{'completed_at': '', 'running_at': '', 'state': 'queued'}
{'completed_at': '', 'running_at': '', 'state': 'running'}
{'completed_at': '2025-11-24T14:17:25.737742Z', 'running_at': '2025-11-24T14:17:24.146228Z', 'state': 'completed'}


<a id='section4'></a>
## 4. Cleanup 

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.3/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id='section5'></a>
## 5. Summary

You successfully completed this notebook! You learned how to create and deploy a R script, and create and run a job using Watson Machine Learning.

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 


### Author

**Jakub Żywiecki**, Software Engineer

Copyright © 2024-2025 IBM. This notebook and its source code are released under the terms of the MIT License.